In [100]:
import pandas as pd
import numpy as np


In [113]:
gas_price = pd.read_csv("data/gas_prices.csv")
gas_price = gas_price[gas_price["product-name"] == "Conventional Regular Gasoline"].drop(columns = ["product-name", "product", "process", "process-name", "series", "series-description"])

gas_state_mapping = {
    "YCLE": ["OH"],
    "Y48SE": ["WA"],
    "R5XCA": ["WA", "OR", "NV", "AZ", "AK", "HI"],
    "R1Z": ["WV", "VA", "NC", "SC", "GA", "FL"],
    "SCO": ["CO"],
    "R30": ["NM", "TX", "AR", "LA", "MS", "AL"],
    "R50": ["WA", "OR", "NV", "CA", "AZ", "AK", "HI"],
    "YMIA": ["FL"],
    "R1Y": ["NY", "PA", "MD", "NJ", "DE"],
    "SOH": ["OH"],
    "R40": ["MT", "ID", "WY", "UT", "CO"],
    "SWA": ["WA"],
    "R10": ["ME", "VT", "NH", "MA", "CT", "RI","NY", "PA", "MD", "NJ", "DE", "WV", "VA", "NC", "SC", "GA", "FL"],
    "R1X": ["ME", "VT", "NH", "MA", "CT", "RI"],
    "SFL": ["FL"],
    "YDEN": ["CO"],
    "SNY": ["NY"],
    "STX": ["TX"],
    "NUS": ["US"],
    "SMN": ["MN"],
    "R20": ["ND", "SD", "NE", "KS", "OK", "MO", "IA","MN", "WI", "IL", "IN", "MI", "OH", "KY", "TN"]
}
gas_price["stateid"] = gas_price["duoarea"].map(gas_state_mapping)
gas_price = gas_price.explode("stateid")
gas_price = gas_price.groupby(["period", "stateid", "units"], as_index=False)["value"].mean()
gas_price.head()

,period,stateid,units,value
0,2019-09,AK,$/GAL,3.018
1,2019-09,AL,$/GAL,2.271
2,2019-09,AR,$/GAL,2.271
3,2019-09,AZ,$/GAL,3.018
4,2019-09,CA,$/GAL,3.018


In [112]:
kwh_price = pd.read_csv("data/kwh_prices.csv")
kwh_price.head()

,period,stateid,stateDescription,sectorid,sectorName,price,price-units
0,2020-12,AK,Alaska,ALL,all sectors,19.01,cents per kilowatt-hour
1,2020-12,AK,Alaska,COM,commercial,19.01,cents per kilowatt-hour
2,2020-12,AK,Alaska,IND,industrial,13.97,cents per kilowatt-hour
3,2020-12,AK,Alaska,OTH,other,0.00,cents per kilowatt-hour
4,2020-12,AK,Alaska,RES,residential,21.64,cents per kilowatt-hour


In [114]:
kwh_price = pd.read_csv("data/kwh_prices.csv")
kwh_price = kwh_price[kwh_price["sectorid"] == "COM"].drop(columns = ["stateDescription","sectorid", "sectorName"])
kwh_price["price"] = kwh_price["price"]/100
kwh_price["price-units"] = "$/kWh"
pricing = pd.merge(kwh_price, gas_price, on=["period", "stateid"], how="inner")
pricing.rename(columns={"price": "electricity_price", "value": "gas_price", "price-units": "electricity_units", "units": "gas_units"}, inplace=True)
pricing.to_csv("pricing.csv", index=False)

C:\Users\Admin\AppData\Local\Temp\ipykernel_25768\1009917699.py:6: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  pricing.rename(columns={"price": "electricity_price", "value": "gas_price", "price-units": "electricity_units", "units": "gas_units"}, inplace=True)


In [115]:
mileage = pd.read_csv("data/mileages-sample.csv").drop(columns=["Make Id", "Model Id", "Submodel Id","Mileage Id","Mileage Epa City Mpg","Mileage Epa Highway Mpg","Mileage Range City","Mileage Range Highway","Mileage Epa City Mpg Electric","Mileage Epa Highway Mpg Electric","Mileage Epa Time To Charge Hr 240v Electric", "Mileage Created", "Mileage Modified", "Mileage Range Electric"])
msrp = pd.read_csv("data/trims-sample.csv").drop(columns = ["Make Id", "Model Id", "Submodel Id", "Trim Description", "Trim Created", "Trim Modified", "Trim Invoice"])
cars = pd.merge(mileage, msrp, how = "inner", on = ["Trim Id", "Model Year", "Make Name", "Model Name", "Submodel Name", "Trim Name"])
cars.head()
cars.to_csv("cars.csv", index=False)

In [116]:
car_pricing = cars.copy()
car_pricing["type"] = np.select(
    [
        car_pricing["Mileage Epa Combined Mpg Electric"].notna(),
        car_pricing["Trim Description"].fillna("").str.contains(r"diesel|tdi|dsl", case=False, na=False) |
        car_pricing["Trim Name"].fillna("").str.contains(r"diesel|tdi|dsl", case=False, na=False) |
        car_pricing["Model Name"].fillna("").str.contains(r"diesel|tdi|dsl", case=False, na=False),
        car_pricing["Trim Description"].fillna("").str.contains(r"hybrid", case=False, na=False) |
        car_pricing["Trim Name"].fillna("").str.contains(r"hybrid", case=False, na=False) |
        car_pricing["Model Name"].fillna("").str.contains(r"hybrid", case=False, na=False),
    ],
    ["electric", "diesel", "hybrid"],
    default="gas"
)
car_pricing = car_pricing[car_pricing["type"] != "diesel"]

avg_1yr_mileage = 13476
avg_5yr_mileage = avg_1yr_mileage * 5
avg_10yr_mileage = avg_1yr_mileage * 10
    
US_AVG_GAS_PRICE = pricing["gas_price"].mean()
US_AVG_ELECTRICITY_PRICE = pricing["electricity_price"].mean()

for index, car in car_pricing.iterrows():
    if car["type"] == "electric":
        car_pricing.loc[index, "5yr_mpg"] = (
            avg_5yr_mileage / car["Mileage Epa Combined Mpg Electric"]
        )
    else:
        car_pricing.loc[index, "5yr_mpg"] = (
            avg_5yr_mileage / car["Mileage Combined Mpg"]
        )

for index, car in car_pricing.iterrows():
    if car["type"] == "electric":
        car_pricing.loc[index, "10yr_mpg"] = (
            avg_10yr_mileage / car["Mileage Epa Combined Mpg Electric"]
        )
    else:
        car_pricing.loc[index, "10yr_mpg"] = (
            avg_10yr_mileage / car["Mileage Combined Mpg"]
        )

for index, car in car_pricing.iterrows():
    if car["type"] == "electric":
        car_pricing.loc[index, "5yr_cost"] = (
            car["5yr_mpg"]/100* car["Mileage Epa Kwh 100 Mi Electric"] * US_AVG_ELECTRICITY_PRICE + car["Trim MSRP"]
        )
    else:
        car_pricing.loc[index, "5yr_cost"] = (
            car["5yr_mpg"] * US_AVG_GAS_PRICE + car["Trim MSRP"]
        )

for index, car in car_pricing.iterrows():
    if car["type"] == "electric":
        car_pricing.loc[index, "10yr_cost"] = (
            car["10yr_mpg"]/100* car["Mileage Epa Kwh 100 Mi Electric"] * US_AVG_ELECTRICITY_PRICE + car["Trim MSRP"]
        )
    else:
        car_pricing.loc[index, "10yr_cost"] = (
            car["10yr_mpg"] * US_AVG_GAS_PRICE + car["Trim MSRP"]
        )

In [119]:
# separating the different car features
trim = car_pricing["Trim Description"].fillna("")
car_pricing["doors"] = trim.str.extract(r"(\d+)\s*dr", expand=False)
car_pricing["body type"] = trim.str.extract(r"\d+\s*dr\s+(.+?)\s*\(", expand=False)
car_pricing["body type"] = car_pricing["body type"].str.split("w/").str[0]
car_pricing["engine liters"] = trim.str.extract(r"(\d+?\.\d*)L", expand=False)
car_pricing["cylinders"] = (trim.str.extract(r"(\d+)cyl", expand=False))
car_pricing["turbo"] = (trim.str.contains(r"turbo", case=False, na=False))
car_pricing["transmission"] = trim.str.extract(r"(\d+\s*[AM]+|CVT|DCT|DSG|AT|MT|Automatic|Manual)", expand=False)

car_pricing.drop(columns=["Trim Description", "Trim Id", "5yr_mpg", "10yr_mpg"], inplace=True)
car_pricing.head(10)

C:\Users\Admin\AppData\Local\Temp\ipykernel_25768\520351558.py:11: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  car_pricing.drop(columns=["Trim Description", "Trim Id", "5yr_mpg", "10yr_mpg"], inplace=True)


,Model Year,Make Name,Model Name,Submodel Name,Trim Name,Mileage Fuel Tank Capacity,Mileage Combined Mpg,Mileage Epa Combined Mpg Electric,Mileage Epa Kwh 100 Mi Electric,Mileage Battery Capacity Electric,Trim MSRP,type,5yr_cost,10yr_cost,doors,body type,engine liters,cylinders,turbo,transmission
0,2020,Acura,ILX,Base,Base,13.2,28.0,NaN,NaN,NaN,25900,gas,31155.473033,36410.946066,4,Sedan,2.4,4,False,8AM
1,2020,Acura,ILX,Base,Premium and A-SPEC Packages,13.2,28.0,NaN,NaN,NaN,29750,gas,35005.473033,40260.946066,4,Sedan,2.4,4,False,8AM
2,2020,Acura,ILX,Base,Premium Package,13.2,28.0,NaN,NaN,NaN,27750,gas,33005.473033,38260.946066,4,Sedan,2.4,4,False,8AM
3,2020,Acura,ILX,Base,Technology and A-SPEC Packages,13.2,28.0,NaN,NaN,NaN,31650,gas,36905.473033,42160.946066,4,Sedan,2.4,4,False,8AM
4,2020,Acura,ILX,Base,Technology Package,13.2,28.0,NaN,NaN,NaN,29650,gas,34905.473033,40160.946066,4,Sedan,2.4,4,False,8AM
5,2020,Acura,MDX,Base,Base,19.5,23.0,NaN,NaN,NaN,49500,gas,55897.967170,62295.934341,4,SUV,3.5,6,False,9A
6,2020,Acura,MDX,Base,Base,19.5,23.0,NaN,NaN,NaN,44500,gas,50897.967170,57295.934341,4,SUV,3.5,6,False,9A
7,2020,Acura,MDX,Base,Base,19.5,23.0,NaN,NaN,NaN,56250,gas,62647.967170,69045.934341,4,SUV,3.5,6,False,9A
8,2020,Acura,MDX,SH-AWD,SH-AWD,19.5,22.0,NaN,NaN,NaN,53500,gas,60188.783860,66877.567720,4,SUV AWD,3.5,6,False,9A
9,2020,Acura,MDX,SH-AWD,SH-AWD,19.5,22.0,NaN,NaN,NaN,58250,gas,64938.783860,71627.567720,4,SUV AWD,3.5,6,False,9A


In [120]:
car_pricing.to_csv("car_pricing.csv", index=False)